# Deploy to Heroku via Colab

Requires:
- Telegram API ID & Hash from my.telegram.org
- Heroku API Key from Account Settings

In [ ]:
#@title Inputs & Deploy
HEROKU_API_KEY = "" #@param {type:"string"}
HEROKU_APP_NAME = "" #@param {type:"string"}
API_ID = "" #@param {type:"string"}
API_HASH = "" #@param {type:"string"}
TELEGRAM_CHANNEL_ID = "" #@param {type:"string"}
USER_SESSION_STRING = "" #@param {type:"string"}
BOT_TOKEN = "" #@param {type:"string"}
API_KEY = "" #@param {type:"string"}

import requests
import random
import string
import time

headers = {
    "Accept": "application/vnd.heroku+json; version=3",
    "Authorization": f"Bearer {HEROKU_API_KEY.strip()}",
    "Content-Type": "application/json"
}

app_name = HEROKU_APP_NAME.strip().lower()
if not app_name:
    suffix = "".join(random.choices(string.ascii_lowercase + string.digits, k=6))
    app_name = f"stremio-tg-{suffix}"
    print(f"Generated name: {app_name}")

addon_url = f"https://{app_name}.herokuapp.com"

payload = {
    "app": {
        "name": app_name
    },
    "source_blob": {
        "url": "https://github.com/SunilRoy-dev/stremio-telegram-debrid/tarball/beta"
    },
    "overrides": {
        "env": {
            "API_ID": API_ID.strip(),
            "API_HASH": API_HASH.strip(),
            "TELEGRAM_CHANNEL_ID": TELEGRAM_CHANNEL_ID.strip(),
            "USER_SESSION_STRING": USER_SESSION_STRING.strip(),
            "BOT_TOKEN": BOT_TOKEN.strip(),
            "API_KEY": API_KEY.strip(),
            "ADDON_URL": addon_url
        }
    }
}

print("Deploying to Heroku...")
r = requests.post("https://api.heroku.com/app-setups", headers=headers, json=payload)

if r.status_code == 202:
    setup_id = r.json()["id"]
    print(f"Deployment started. App URL: {addon_url}")
    
    # wait for build to succeed
    while True:
        status_resp = requests.get(f"https://api.heroku.com/app-setups/{setup_id}", headers=headers)
        if status_resp.status_code == 200:
            data = status_resp.json()
            status = data.get("status")
            print(f"Build status: {status}")
            if status == "succeeded":
                manifest = "/manifest.json"
                if API_KEY.strip():
                    manifest = f"/manifest.json?api_key={API_KEY.strip()}"
                print("\nSuccess! Manifest URL:")
                print(f"{addon_url}{manifest}")
                break
            elif status == "failed":
                print("Deployment failed:", data.get("failure_message"))
                break
        else:
            print(f"Error checking status: {status_resp.text}")
            break
        time.sleep(10)
else:
    print(f"Failed to start deployment: {r.status_code} - {r.text}")
